In [ ]:
# Add tags for subpopulation and analysis purposes
# Attributes:
# - works in brussels: True/False
# - lives in brussels: True/False

# Subpopulation:
# - short_distance -- anyone with a home-work distance < 20 km
# - long_distance -- anyone with a home-work distance >= 20 km
# - company_car -- any resident with a company car

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Load the data
all_workers = pd.read_csv("../9_merge/output/workers_merged.csv")

In [ ]:
all_workers["subpopulation"] = np.select(
    [
        all_workers["has_company_car"].eq(1) & all_workers["lives_in_brussels"].eq(True),
        all_workers["commute_dist_km"].ge(20),
    ],
    [
        "company_car",
        "long_distance",
    ],
    default="short_distance",
)

In [ ]:
subpopulation_stats = (
    all_workers["subpopulation"]
    .value_counts(dropna=False)
    .rename_axis("subpopulation")
    .reset_index(name="count")
)

subpopulation_stats["share_pct"] = (
    subpopulation_stats["count"] / len(all_workers) * 100
)

display(subpopulation_stats)

In [ ]:
all_workers.to_csv("output/all_active_workers_final.csv", index=False)

# Fixing default modes for company_car

In [ ]:
company_car_check = all_workers.loc[
    all_workers["subpopulation"].eq("company_car"),
    ["subpopulation", "assigned_mode"],
]

not_car = company_car_check[~company_car_check["assigned_mode"].eq("car")]

print(f"Total company_car agents: {len(company_car_check)}")
print(f"company_car agents with mode != 'car': {len(not_car)}")

if not_car.empty:
    print("✅ All company_car agents have mode 'car'.")
else:
    print("❌ Some company_car agents do not have mode 'car'.")
    display(not_car.head(10))

In [ ]:
all_workers.loc[all_workers["subpopulation"].eq("company_car"), "assigned_mode"] = "car"

# Fixing default modes for long_distance

In [ ]:
long_distance_check = all_workers.loc[
    all_workers["subpopulation"].eq("long_distance"),
    ["subpopulation", "assigned_mode"],
]

walk = long_distance_check[long_distance_check["assigned_mode"].eq("walk")]

print(f"Total long_distance agents: {len(long_distance_check)}")
print(f"long_distance agents with mode = 'walk': {len(walk)}")

if walk.empty:
    print("✅ All long_distance agents have mode 'walk'.")
else:
    print("❌ Some long_distance agents do not have mode 'walk'.")
    display(walk.head(10))

In [ ]:
mask = all_workers["subpopulation"].eq("long_distance") & all_workers["assigned_mode"].eq("walk")
all_workers.loc[mask, "assigned_mode"] = "pt"

print(f"Updated {mask.sum()} long_distance agents from 'walk' to 'pt'.")

In [ ]:
long_distance_check = all_workers.loc[
    all_workers["subpopulation"].eq("long_distance"),
    ["subpopulation", "assigned_mode"],
]

bike = long_distance_check[long_distance_check["assigned_mode"].eq("bike")]

print(f"Total long_distance agents: {len(long_distance_check)}")
print(f"long_distance agents with mode = 'bike': {len(bike)}")

if bike.empty:
    print("✅ All long_distance agents have either 'car' or 'pt'.")
else:
    print("❌ Some long_distance agents have mode 'bike'.")
    display(bike.head(10))

In [ ]:
mask = all_workers["subpopulation"].eq("long_distance") & all_workers["assigned_mode"].eq("bike")
all_workers.loc[mask, "assigned_mode"] = "pt"

print(f"Updated {mask.sum()} long_distance agents from 'bike' to 'pt'.")

In [ ]:
long_distance_check = all_workers.loc[
    all_workers["subpopulation"].eq("long_distance"),
    ["subpopulation", "assigned_mode"],
]

ebike = long_distance_check[long_distance_check["assigned_mode"].eq("e-bike")]

print(f"Total long_distance agents: {len(long_distance_check)}")
print(f"long_distance agents with mode = 'e-bike': {len(ebike)}")

if ebike.empty:
    print("✅ No long_distance agents have mode 'e-bike'.")
else:
    print("❌ Some long_distance agents have mode 'e-bike'.")
    display(ebike.head(10))

In [ ]:
mask = all_workers["subpopulation"].eq("long_distance") & all_workers["assigned_mode"].eq("e-bike")
all_workers.loc[mask, "assigned_mode"] = "pt"

print(f"Updated {mask.sum()} long_distance agents from 'e-bike' to 'pt'.")

In [ ]:
long_distance_modal_split = (
    all_workers.loc[all_workers["subpopulation"].eq("long_distance"), "assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("assigned_mode")
    .reset_index(name="count")
)

long_distance_modal_split["share_pct"] = (
    long_distance_modal_split["count"] / long_distance_modal_split["count"].sum() * 100
)

display(long_distance_modal_split)

In [ ]:
# Modal split by commute distance bands
distance_bins = [0, 2, 5, 15, 30, 50, np.inf]
distance_labels = ["0-2", "2-5", "5-15", "15-30", "30-50", "50+"]

tmp = all_workers[["commute_dist_km", "assigned_mode"]].dropna().copy()
tmp["distance_band"] = pd.cut(
    tmp["commute_dist_km"],
    bins=distance_bins,
    labels=distance_labels,
    right=False,
)

# Absolute counts by distance band and mode
mode_by_distance_counts = pd.crosstab(tmp["distance_band"], tmp["assigned_mode"])

# Percent shares within each distance band
mode_by_distance_share = (
    pd.crosstab(tmp["distance_band"], tmp["assigned_mode"], normalize="index") * 100
).round(2)

print("Counts:")
print(mode_by_distance_counts)

print("\nShare (%) within each distance band:")
print(mode_by_distance_share)

# Optional quick visualization
mode_by_distance_share.plot(kind="bar", stacked=True, figsize=(10, 5))

# SHORT DISTANCE

In [ ]:
short_distance_modal_split = (
    all_workers.loc[all_workers["subpopulation"].eq("short_distance"), "assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("assigned_mode")
    .reset_index(name="count")
)

short_distance_modal_split["share_pct"] = (
    short_distance_modal_split["count"] / short_distance_modal_split["count"].sum() * 100
)

display(short_distance_modal_split)

# Distance > 10 km and walking

In [ ]:
walk_over_10km = all_workers[
    (all_workers["commute_dist_km"] > 10) & (all_workers["assigned_mode"].eq("walk"))
].copy()

print(f"Agents with commute_dist_km > 10 and assigned_mode == 'walk': {len(walk_over_10km)}")

In [ ]:
mask = all_workers["commute_dist_km"].gt(10) & all_workers["assigned_mode"].eq("walk")
all_workers.loc[mask, "assigned_mode"] = "bike"

print(f"Updated {mask.sum()} > 10km agents from 'walk' to 'bike'.")

# FINAL CHECK

In [ ]:
# Filter on the ones who also work in Brussels
workers_in_brussels = all_workers[all_workers["works_in_brussels"]].copy()

commuters = all_workers[~all_workers["lives_in_brussels"]].copy()
residents = all_workers[all_workers["lives_in_brussels"]].copy()

In [ ]:
# Split of transport mode
print("All")
mode_split = (
    all_workers["assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

# Only people who work in Brussels
print("\nWork in Brussels")
mode_split = (
    workers_in_brussels["assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

# Only Brussels residents
print("\nResidents")
mode_split = (
    residents["assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

# Only Brussels commuters
print("\nCommuters")
mode_split = (
    commuters["assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

In [ ]:
all_workers.to_csv("output/all_active_workers_final.csv", index=False)